In [ ]:
# ============================================================
# CONFIGURAÇÃO — ajuste estas variáveis para o seu ambiente
# ============================================================
import os

# Caminhos para os arquivos CSV do seu dataset
TRAINING_CSV    = os.environ.get('TRAINING_CSV',    '../truths/sample_data.csv')
FULL_DATASET_CSV = os.environ.get('FULL_DATASET_CSV', '../truths/sample_data.csv')
VALIDATION_CSV  = os.environ.get('VALIDATION_CSV',  '../truths/sample_data.csv')

# Nomes das colunas no CSV
TRANSCRIPTION_COLUMN = 'transcription'
LABEL_COLUMN         = 'Assunto tratado'  # coluna com lista de labels (string ou lista Python)
AUDIO_COLUMN         = 'audios'

# Colunas que contêm nomes de pessoas — serão adicionadas às stopwords
# Deixe lista vazia [] se o seu dataset não tiver essas colunas
NAME_COLUMNS = os.environ.get('NAME_COLUMNS', '').split(',') if os.environ.get('NAME_COLUMNS') else []

# Diretório para salvar embeddings intermediários (cache)
EMBEDDINGS_DIR = os.environ.get('EMBEDDINGS_DIR', 'embeddings_save/')

# Diretório base para salvar e carregar modelos
MODELS_DIR = os.environ.get('MODELS_DIR', '../models/')

# Categoria catch-all (mutuamente exclusiva com as demais)
CATCHALL_CATEGORY = os.environ.get('CATCHALL_CATEGORY', 'Outros assuntos')

# Threshold padrão de classificação (ajuste por classe na célula de validação)
DEFAULT_THRESHOLD = float(os.environ.get('DEFAULT_THRESHOLD', '0.5'))
# ============================================================

# Data preparation

In [ ]:
import pandas as pd
import ast
import re

df = pd.read_csv(TRAINING_CSV)

def garantir_lista(valor):
    if isinstance(valor, list):
        return valor
    if isinstance(valor, str):
        try:
            return ast.literal_eval(valor)
        except:
            return [valor]
    return []

df[LABEL_COLUMN] = df[LABEL_COLUMN].apply(garantir_lista)
df = df[[AUDIO_COLUMN, TRANSCRIPTION_COLUMN, LABEL_COLUMN]]
df

In [ ]:
valores_remover = [CATCHALL_CATEGORY]

def limpar_assuntos(lista):
    if len(lista) == 1 and lista[0] in valores_remover:
        return None
    return [x for x in lista if x not in valores_remover]

df[LABEL_COLUMN] = df[LABEL_COLUMN].apply(limpar_assuntos)
df = df[df[LABEL_COLUMN].notna()]
df = df[df[LABEL_COLUMN].apply(lambda x: len(x) > 0)]
df

# Embedding

## Stop words

In [ ]:
import pandas as pd
import json, ast, re
import unidecode

def clean_transcription(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'_x000D_', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'x000d', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[#\$]*_x000d_+', ' ', text, flags=re.IGNORECASE)
    text = text.lower()
    return text

stopwords_nomes_proprios = set()

# Carrega o dataset completo apenas se houver colunas de nomes configuradas
if NAME_COLUMNS:
    d = pd.read_csv(FULL_DATASET_CSV)

    def parse_nome(val):
        if not isinstance(val, str):
            return []
        val = val.strip()
        if val.startswith('{') and 'text' in val:
            try:
                parsed = json.loads(val.replace("'", '"'))
                if isinstance(parsed, dict) and 'text' in parsed:
                    tokens = []
                    for item in parsed['text']:
                        tokens.extend(item.strip().split())
                    return tokens
            except Exception:
                pass
        return val.split()

    for col in NAME_COLUMNS:
        if col in d.columns:
            d[col] = d[col].apply(clean_transcription)
            for val in d[col].dropna().astype(str):
                tokens = parse_nome(val)
                tokens = [unidecode.unidecode(t.lower()) for t in tokens]
                stopwords_nomes_proprios.update(tokens)

print(f'Stopwords de nomes próprios extraídas: {len(stopwords_nomes_proprios)}')
print('Exemplo:', list(stopwords_nomes_proprios)[:10])

In [ ]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stopwords_pt = stopwords.words("portuguese")

stopwords_sociais = ["tchau","ate","bom","oi","beleza","valeu","obrigado","obrigada","joia","tranquilo",
 "falou","cara","opa","boa","dia","tarde","noite","perfeito","entao"]

stopwords_funcionais_pt = ["de","do","da","das","dos","para","por","com","em","ao","aos","as","os",
 "um","uma","uns","umas","se","so","mas","que","ne","isso","nao","sim","ai",
 "la","ja","agora","bem","muito","pode","vai","vou","vamos","tem","tudo",
 "nada","tambem","mesmo","porque","entendi","acho","ainda","ficar","caso"]

stopwords_contexto_operacional = ["centro","sudeste","norte","sul","oeste","nacional","regional","ons"]

stopwords_pt.extend(stopwords_sociais)
stopwords_pt.extend(stopwords_funcionais_pt)
stopwords_pt.extend(stopwords_contexto_operacional)
stopwords_pt.extend(stopwords_nomes_proprios)

stopwords_pt = list(set(stopwords_pt))

print(stopwords_pt)  # só para inspecionar

## Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from openai_embedding_transformer import OpenAIEmbeddingTransformer

ct = ColumnTransformer(
    transformers=[
        ("embed", OpenAIEmbeddingTransformer(model="text-embedding-3-small"), "transcription"),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 5),
            max_features=30000,  # ajuste conforme o tamanho do corpus
            stop_words=stopwords_pt
        ), "transcription"),
    ],
    # Mantém saída esparsa se possível (útil p/ TF-IDF grande).
    # Os embeddings (densos) são convertidos para esparso ao concatenar.
    sparse_threshold=1.0,
    remainder="drop",
    # opcional: ponderar blocos (ex.: embeddings com peso 2x)
    transformer_weights={"embed": 1.5, "tfidf": 2.0},
)

## Separação dataset

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import ast, json

# Normaliza a coluna para listas de labels
def normalizar_assunto_tratado(texto):
    # Se já for lista, mantém
    if isinstance(texto, list):
        return texto
    
    if not isinstance(texto, str):
        return []
    
    try:
        data = json.loads(texto)
        if isinstance(data, dict) and "choices" in data:
            return data["choices"]
    except:
        pass
    
    try:
        data = ast.literal_eval(texto)
        if isinstance(data, list):
            return data
    except:
        pass
    
    return [texto.strip()]

df = df.copy()

df["labels"] = df["Assunto tratado"].apply(normalizar_assunto_tratado)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["labels"])

print("Classes multilabel:", mlb.classes_)
print("Shape do y:", y.shape)

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit

X = df["transcription"].astype(str)   # garante string

# Faz split estratificado multilabel
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.12, random_state=42)

for train_idx, test_idx in msss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

# Verifica distribuição
import numpy as np
print("Distribuição treino:", np.sum(y_train, axis=0))
print("Distribuição teste:", np.sum(y_test, axis=0))


## Criação dos embeddings

In [ ]:
ct.fit(X_train.to_frame())

X_train_features = ct.transform(X_train.to_frame())
X_test_features  = ct.transform(X_test.to_frame())

In [ ]:
from joblib import dump, load

dump(X_train_features, "embeddings_save/X_train_features.joblib")
dump(X_test_features, "embeddings_save/X_test_features.joblib")
dump(y_train, "embeddings_save/y_train.joblib")
dump(y_test, "embeddings_save/y_test.joblib")

# Salva apenas o tfidf (funciona com pickle/joblib)
tfidf = ct.named_transformers_['tfidf']
dump(tfidf, "embeddings_save/tfidf_vectorizer.joblib")

## Load dos embeddings (caso não queira criá-los novamente)

In [ ]:
# Depois você recria o ColumnTransformer assim:
from joblib import load
tfidf_loaded = load("embeddings_save/tfidf_vectorizer.joblib")

from sklearn.compose import ColumnTransformer
ct_reload = ColumnTransformer(
    transformers=[
        ("embed", OpenAIEmbeddingTransformer(model="text-embedding-3-small"), "transcription"),
        ("tfidf", tfidf_loaded, "transcription"),
    ],
    sparse_threshold=1.0,
    remainder="drop"
)

## Análise dos embeddings

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import TruncatedSVD

# Reduz TF-IDF para acelerar
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X_train_features)

import umap
import matplotlib.pyplot as plt
import numpy as np

umap_model = umap.UMAP(
    n_components=3,
    random_state=42,
    n_neighbors=30,
    min_dist=0.0,   # clusters mais definidos
    metric="cosine" # melhor para embeddings textuais
)
X_3d_umap = umap_model.fit_transform(X_reduced)

y_labels = [mlb.classes_[np.argmax(row)] for row in y_train]

In [ ]:
import plotly.express as px
import pandas as pd

df_3d = pd.DataFrame(X_3d_umap, columns=["x", "y", "z"])
df_3d["label"] = y_labels

fig = px.scatter_3d(
    df_3d, x="x", y="y", z="z",
    color="label",
    opacity=0.7,
    title="Distribuição 3D dos textos (UMAP)"
)
fig.show()


# Model

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.multioutput import ClassifierChain

lgbm = LGBMClassifier(
    objective="binary",   # importante para multilabel
    class_weight="balanced",
    random_state=42,
    n_estimators=250,
    learning_rate=1,
    max_depth=7,
    num_leaves=20,
    n_jobs=-1,
    min_data_in_leaf=1,
    verbose=-1,
)

clf = ClassifierChain(
    base_estimator=lgbm, 
    order='random', 
    random_state=42,
)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain

svm = LinearSVC(
    class_weight='balanced', 
    random_state=42,
    C=1,
    multi_class="ovr",
)

clf = ClassifierChain(base_estimator=svm, order='random', random_state=42)

In [ ]:
from sklearn.svm import SVC
from sklearn.multioutput import ClassifierChain

svm = SVC(
    probability=True, 
    kernel="linear", 
    class_weight="balanced", 
    random_state=42
)

clf = ClassifierChain(
    base_estimator=svm, 
    order='random', 
    random_state=42
)

# Pipeline training

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline(
    steps=[
        ("features", ct),  # gera [embeddings + tfidf]
        ("clf", clf),
    ]
)

pipe.fit(X_train.to_frame(name="transcription"), y_train)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = pipe.predict(X_test.to_frame(name="transcription"))

# Avaliação
print("Acurácia (exata):", accuracy_score(y_test, y_pred))  # só acerta se TODAS labels da amostra baterem

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=mlb.classes_))

In [ ]:
from joblib import dump
import os

model_name = input('Nome do modelo (ex: svm_v1): ')
model_dir = os.path.join(MODELS_DIR, model_name)
os.makedirs(model_dir, exist_ok=True)

dump(pipe, os.path.join(model_dir, 'pipeline_embeddings_tfidf_svm.joblib'))
dump(np.array(mlb.classes_), os.path.join(model_dir, 'classes.npy'))
print(f'Modelo salvo em: {model_dir}')

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

clf.fit(X_train_features, y_train)

y_pred = clf.predict(X_test_features)

# Avaliação
print("Acurácia (exata):", accuracy_score(y_test, y_pred))  # só acerta se TODAS labels da amostra baterem

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=mlb.classes_))

# Pipeline

In [ ]:
from joblib import load
import numpy as np
import os

model_name = input('Nome do modelo a carregar (ex: svm_v1): ')
model_dir = os.path.join(MODELS_DIR, model_name)

pipe   = load(os.path.join(model_dir, 'pipeline_embeddings_tfidf_svm.joblib'))
classes = load(os.path.join(model_dir, 'classes.npy'))
pipe

In [ ]:
import pandas as pd

# Substitua pelo texto que deseja classificar
transcription = input('Digite a transcrição para classificar: ')

teste_1 = pd.DataFrame({'transcription': [transcription]})

probas = pipe.predict_proba(teste_1)

threshold = DEFAULT_THRESHOLD
pred_bin = (probas[0] >= threshold).astype(int)
predicoes = [classes[i] for i, val in enumerate(pred_bin) if val == 1]

print('Probabilidades:', np.round(probas[0], 3))
print('Classes previstas:', predicoes if predicoes else [CATCHALL_CATEGORY])

# Teste

## Main

In [ ]:
import pandas as pd

df_validation = pd.read_csv(VALIDATION_CSV)
df_validation = df_validation[[AUDIO_COLUMN, TRANSCRIPTION_COLUMN, LABEL_COLUMN]]
df_validation

In [ ]:
import pandas as pd
import re
import unicodedata
import ast
import json

def normalizar_texto(texto: str) -> str:
    """
    Função padronizada para normalização de texto para cálculo de métricas.
    Converte para minúsculas, remove acentos e pontuação, e filtra palavras de uma letra.
    """
    if not isinstance(texto, str) or pd.isna(texto):
        return ""
    
    # Converte para minúsculas
    texto = texto.lower()
    
    # Remove acentuação
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join([c for c in texto if unicodedata.category(c) != 'Mn'])
    
    # Remove pontuação mantendo apenas letras, números e espaços
    texto = re.sub(r'[^\w\s]', '', texto)
    
    # Remove múltiplos espaços
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    # Remove palavras de uma letra
    texto = ' '.join([word for word in texto.split() if len(word) > 1])
    
    return texto

def normalizar_assunto_tratado(texto):
    # garante que texto é string
    if not isinstance(texto, str):
        return []
    
    try:
        # tenta carregar como JSON
        data = json.loads(texto)
        if isinstance(data, dict) and "choices" in data:
            return data["choices"]
    except (json.JSONDecodeError, TypeError):
        pass
    
    try:
        # tenta carregar como lista escrita em string (ex: "['a','b']")
        data = ast.literal_eval(texto)
        if isinstance(data, list):
            return data
    except (ValueError, SyntaxError):
        pass
    
    # caso seja string simples, embrulha numa lista
    return [texto.strip()]

df_validation['transcription'] = df_validation['transcription'].apply(normalizar_texto)

# Transforma os valores da coluna em listas
df_validation['Assunto tratado'] = df_validation['Assunto tratado'].apply(normalizar_assunto_tratado)
df_validation

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# Configure thresholds por classe — ajuste após análise das métricas de validação.
# Threshold mais baixo → mais sensível (recall maior, precision menor).
# Threshold mais alto  → mais conservador (precision maior, recall menor).
#
# Exemplo (substitua pelos nomes das SUAS classes e valores calibrados):
#
# thresholds = {
#     'CLASSE_RARA_1': 0.2,   # classes raras: threshold menor para recuperar mais
#     'CLASSE_RARA_2': 0.3,
#     'CLASSE_MEDIA':  0.5,
#     'CLASSE_GRANDE': 0.65,  # classes frequentes: threshold maior p/ evitar FP
# }
#
# Se não configurar, usa DEFAULT_THRESHOLD para todas as classes.
thresholds = {}  # preencha com suas classes após avaliar o modelo


def main(transcription):
    teste = pd.DataFrame({'transcription': [transcription]})
    probas = pipe.predict_proba(teste)

    pred_bin = np.zeros_like(probas, dtype=int)
    for i, cls in enumerate(classes):
        t = thresholds.get(cls, DEFAULT_THRESHOLD)
        pred_bin[:, i] = (probas[:, i] >= t).astype(int)

    predicoes = [classes[i] for i, val in enumerate(pred_bin[0]) if val == 1]
    return predicoes if predicoes else [CATCHALL_CATEGORY]


preds = []
df_result = df_validation.copy()

for idx, row in df_validation.iterrows():
    print(f'\nProcessando {idx}...')
    print(f'  transcrição: {row[TRANSCRIPTION_COLUMN]}')
    print(f'  esperado:    {row[LABEL_COLUMN]}')
    subj = main(row[TRANSCRIPTION_COLUMN])
    print(f'  obtido:      {subj}')
    preds.append(subj)

df_result['A'] = preds

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score

def ensure_list(val):
    if isinstance(val, list):
        return [str(x).strip() for x in val]
    if pd.isna(val):
        return []
    return [str(val).strip()]

# garantir listas
df_result["Assunto tratado"] = df_result["Assunto tratado"].apply(ensure_list)
df_result["A"]   = df_result["A"].apply(ensure_list)

# binarizar
mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df_result["Assunto tratado"])
y_pred = mlb.transform(df_result["A"])

# métricas
print("Subset Accuracy:", accuracy_score(y_true, y_pred))
print("F1 micro:", f1_score(y_true, y_pred, average="micro"))
print("F1 macro:", f1_score(y_true, y_pred, average="macro"))
print("F1 weighted:", f1_score(y_true, y_pred, average="weighted"))
print("F1 samples:", f1_score(y_true, y_pred, average="samples"))
print("Relatório de classificação:")
print(classification_report(y_true, y_pred, target_names=mlb.classes_))